# Paper figures

Figure panels of *Agent-Based Model with Decaying Attention Reproducing Hashtag Scaling Laws: Random Multiplicative Growth as a Bridge*. Each panel is written as a separate PDF; the multi-panel figures of the paper are assembled from them in a vector editor.

Check the paths in the next cell, run the four setup cells, then run whichever figure section you need. The sections are independent of each other, except that Fig. 4 and Fig. 6 each begin with a cell that loads their simulation run, and Fig. 7 reuses the Model 1 run loaded in the Fig. 6 section.

**Not included here:** Fig. 3 (TikZ schematic, see `conceptual/`), Fig. 8 (dataset description, drawn from the raw tweet and network tables) and Fig. 9b (error heat map of the earlier calibration grid). None of these changes when the simulations are re-run.

In [ ]:
# ============================== configuration ==============================
# Every path used by this notebook is set here. Defaults assume the layout described in
# ../data/README.md (tables from the Zenodo deposit placed inside this repository) and the
# simulation output produced by ../simulation/parallel_sim.sh. Adjust if your files live elsewhere.
BASE       = '..'                                   # repository root (code_github/)
S          = BASE + '/simulation/simulation_results/'     # simulation output: model0/, model1/
R          = BASE + '/results/'                           # derived tables (new users, expected counts)
ANALYSIS   = BASE + '/analysis'                           # python modules
REAL       = BASE + '/simulation/real_data/'              # empirical tables
FIGS       = './output/'                                  # where the panel PDFs are written
LCC10MIN   = REAL + 'hashtag_counts_10min_lcc.pkl'
NOISE10MIN = REAL + 'hashtag_counts_10min_lcc_noise.pkl'   # optional, not used in the paper
HOURLY     = REAL + 'hashtag_counts_hourly.pkl'

# all comparison statistics use the final N_DAYS_CMP model days (Methods: "final 7 model days"),
# matching the 7-day empirical window
N_DAYS_CMP = 7

# the two runs behind Fig. 4 and Fig. 6
RUN0 = 'N397369_T30_HOUR93626_pn0.0218_constdk_c1_real_net_m24_steps_SEED1'
RUN1 = 'N397369_T30_HOUR93626_pn0.0218_ou_sigma0.1_theta0.007_mu0_d013_real_net_m84_steps_SEED1'

import os
os.makedirs(FIGS, exist_ok=True)
print('S    =', S)
print('FIGS =', FIGS)

In [ ]:
# ---- preflight: which input files are present? ----------------------------
import os
NEEDED = {
    'Fig 1':            [HOURLY],
    'Fig 2 (a-d, f)':   [REAL + 'hashtag_counts_daily.pkl'],
    'Fig 2e / 4c / 6e': [R + 'new_adopters_daily.pkl'],
    'Fig 4':            [S + f'model0/results/{RUN0}.pkl',
                         S + f'model0/new_users/{RUN0}.txt'],
    'Fig 5':            [LCC10MIN, R + 'model0_expected_counts_10min.pkl'],
    'Fig 6 / 7':        [S + f'model1/results/{RUN1}.pkl',
                         S + f'model1/new_users/{RUN1}.txt'],
    'Fig 9a':           [S + f'model1/diffu_value/sample{RUN1}.txt'],
}
missing = []
for fig, paths in NEEDED.items():
    bad = [q for q in paths if not os.path.exists(q)]
    print(('  OK   ' if not bad else '  MISS ') + fig)
    for q in bad:
        print('         ' + q); missing.append(q)
if missing:
    print(f'\n{len(missing)} file(s) missing -- the sections marked MISS will fail; the others are fine.')
else:
    print('\nall inputs present.')


In [ ]:
import sys, os
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns
from tqdm import trange
from tqdm.notebook import tqdm
from matplotlib.ticker import LogLocator

sys.path.append(ANALYSIS)
from result_analysis import (plot_cdf, plot_growth_rate_cdf, plot_realdata_usage, plot_simulated_usage,
                             growthRateDistribution, compare_log_growth_pdf, trend_follow,
                             growth_rate_scaling_errorbar_plot, growth_rate_scaling_errorbar_prepare_data,
                             next_day_analysis_new, log_rmse, rmse, batch_autocorr,
                             visulize_diffusion_series_paper, potention_analysis_paper, plot_potential)
from new_adopters import simulated_new_adopters, adopter_pairs, plot_adopter_scaling
from random_multiplicative_process import random_simulation
import matplotlib.transforms as mtransforms

mpl.rcParams['axes.unicode_minus'] = False
plt.style.use('default')
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'Times New Roman'
plt.rcParams["legend.framealpha"] = 0.2
plt.rcParams["legend.frameon"] = True
sns.set_style("ticks")
sns.set_context("notebook", font_scale=1.5,
                rc={'font.size': 10, 'axes.labelsize': 35, 'legend.fontsize': 10,
                    'xtick.labelsize': 25, 'ytick.labelsize': 25, 'figure.titlesize': 35})
tqdm.pandas()
%matplotlib inline
%load_ext autoreload
%autoreload 2

def save(name, pad=0.05):
    """Write the current figure to FIGS/name (300 dpi, tight bounding box)."""
    plt.savefig(FIGS + name, bbox_inches='tight', pad_inches=pad, dpi=300)

In [ ]:
# empirical tables
hashtag_df = pd.read_pickle(REAL + 'hashtag_counts_daily.pkl')      # daily counts, hashtag x date
real_bt    = np.log10(hashtag_df.shift(-1, axis=1) / hashtag_df)

# new adopters (called "new users" in the code); only needed by Fig. 2e / 4c / 6e
try:
    emp_new = pd.read_pickle(R + 'new_adopters_daily.pkl')
except FileNotFoundError:
    emp_new = None
    print('note: new_adopters_daily.pkl not found -- Fig. 2e / 4c / 6e cannot be drawn.')
    print('      rebuild it with analysis/new_adopters.py --posts <raw post table> ')

print(hashtag_df.shape, None if emp_new is None else emp_new.shape)


In [ ]:
# shared helper: (x_k(t), n_k^new(t+1)) pairs of a simulation run, final N_DAYS_CMP days
FIT = dict(y_min=1, fit_xmin=5, min_count=10, alpha_digits=2,
           xlim=(0.7, 10**5.4), ylim=(0.7, 10**5))

def sim_pair(model, name):
    sim  = pd.read_pickle(S + f'{model}/results/{name}.pkl')
    snew = simulated_new_adopters(S + f'{model}/new_users/{name}.txt')
    keep = [j for j in range(min(sim.shape[1], snew.shape[1])) if sim.iloc[:, j].notna().any()]
    keep = keep[-N_DAYS_CMP:]
    return adopter_pairs(sim.iloc[:, keep], snew.iloc[:, keep])

## Fig. 1 — example usage time series

Needs the hourly count table `hashtag_df_hour`.

In [ ]:
hashtag_df_hour = pd.read_pickle(HOURLY)          # hourly counts
hashtag_df_hour = hashtag_df_hour.loc[hashtag_df.index]
print(hashtag_df_hour.shape)

In [ ]:
name_ls = ['jishin', 
           'genpatu',
           "edano_nero",
           'nowplaying',
           ]
name_translated = [
    "(earthquake)",
    "(nuclear power plant)",
    "(Mr. Edano, please get some rest)",
    "(a music-sharing tag)"
]
plt.figure(figsize=(18 * 1.2, 4 * len(name_ls)))
for cnt, name in enumerate(name_ls):
    ax1 = plt.subplot(len(name_ls), 2, cnt+1)
    ax1.plot(hashtag_df_hour.loc[name, :], '-X', linewidth=4, markersize=8, label = r'$x_k(h)$ (left axis)')
    if cnt == 2 or cnt == 3: ax1.set_xticks([hashtag_df_hour.columns[i] for i in np.arange(0, 24*7, 24) + 10], [f'3/{i}' for i in range(11, 18)])
    else: ax1.set_xticks([hashtag_df_hour.columns[i] for i in np.arange(0, 24*7, 24) + 10], [])
    plt.title(f'{name} {name_translated[cnt]}', fontsize = 25)    
    for i in range(1, 7, 1): ax1.axvline(i * 24, color='gray', linestyle='--', alpha=0.5)
    for i in range(1, 7, 2): plt.axvspan(24*(i), 24*(i+1), color = 'grey', alpha = 0.1)
    width = 3
    ax1.set_xlim(0 - width, 24*7 + width);
    ax2 = ax1.twinx()
    ax2.plot(hashtag_df_hour.columns[12::24], hashtag_df.loc[name, :], '-X', c = 'r', label = r'$x_k(t)$ (right axis)', alpha = 0.5, linewidth = 8, markersize = 16)
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    if cnt == 0:
        ax1.legend(h1+h2, l1+l2, fontsize = 22, loc = 'upper right', framealpha=0)
    if cnt == 3: ax1.set_ylim(bottom = -20)
    if cnt == 0:
        t_eq = 14 + 46/60
        trans = mtransforms.blended_transform_factory(ax1.transData, ax1.transAxes)
        ax1.plot([t_eq], [0], marker='^', markersize=16, color='black',
                transform=trans, clip_on=False, zorder=6)
        ax1.text(t_eq + 2.5, 0.04, 'main shock (14:46 JST)',
            transform=trans, fontsize=15, ha='left', va='bottom')
    
plt.tight_layout()
save('real_series_example_v2.pdf')

## Fig. 2 — the five empirical regularities

(a) usage distribution, (b) growth-rate PDF, (c) conditional CDF, (d) size dependence of the growth rate, (e) next-day new users, (f) deactivation probability. All six are empirical only and do not depend on the simulations.

In [ ]:
plt.figure(figsize = (8, 6))
for i in range(hashtag_df.shape[1]):
    plot_cdf(hashtag_df.iloc[:, i], c = f'C{i}', label = '3/{}'.format(i+11))
ax = plt.gca()
ax.xaxis.set_minor_locator(LogLocator(subs='all', numticks=100))
plt.legend(fontsize = 20, frameon = False)
plt.xlabel(r'$x(t)$')
plt.ylabel(r'$P(\geq x(t))$')
x = np.arange(10, 10**5)
alpha = -0.8
y = 0.25 * (x/min(x))**alpha
plt.plot(x, y, '--', c='k', linewidth = 3)
plt.grid(False)
plt.tight_layout()
save('real_hashtag_cdf.pdf')

In [ ]:
plt.figure(figsize=(8, 6))
compare_log_growth_pdf(real_bt, real_bt, real_label = None, simu_bar = False)
plt.xticks(range(-4, 5))
plt.ylabel(r'$p(\log b(t))$')
save('real_growth_rate_pdf.pdf')

In [ ]:
paras = growthRateDistribution(hashtag_df.fillna(0).T,
                               stationaryPoint = 0, base = 10, step = 0.5, 
                               minmal_sample = 80, 
                               cumulated_norm = False,
                               plot_jump_sample = 1, 
                               show_detail = False, 
                               figsize=(10.5, 6))
ax = plt.gca()
ax.xaxis.set_minor_locator(LogLocator(subs='all', numticks=100))
plt.legend(fontsize=20, ncol=1, bbox_to_anchor=(1.0, 0.1), loc=3, borderaxespad=0, frameon=False)
plt.tight_layout()
save('real_growth_rate_conditional_cdf.pdf')

In [ ]:
std_ge_result_real, std_ne_result_real = growth_rate_scaling_errorbar_prepare_data(hashtag_df.fillna(0).T, 
                                        stationaryPoint = 0, base = 20, step = 0.5, minmal_sample=30, show_detail = False)
plt.figure(figsize=(8, 6))
growth_rate_scaling_errorbar_plot(std_ge_result_real, std_ne_result_real, label = '', c = ['C0', 'C0'])
plt.legend(fontsize=20, frameon = False)
plt.tight_layout()
save('real_growth_rate_scaling_errorbar.pdf')

In [ ]:
# Fig. 2(e): empirical data only; the dashed fit carries the alpha label
plt.figure(figsize=(8, 6))
plot_adopter_scaling(*adopter_pairs(hashtag_df, emp_new), label=None, c='C0', marker='o',
                     fit=True, alpha_in_label=False, **FIT)
save('real_trend_follow_v2.pdf')

In [ ]:
plt.figure(figsize=(8, 6))
result_errbar = next_day_analysis_new(hashtag_df, logy = True, minvalue=0, 
                                      label = None, c = 'C0', 
                                      error_line = False, fit_line = True,
                                      return_=False)
plt.tight_layout()
save('real_prob_of_oneday_death.pdf')

## Fig. 4 — Model 0

Panels (a)–(d) come from the m = 24 main run, (e) from the three randomised networks, (f) from the memory-window sweep. Run the loader cell first.

In [ ]:
# Model 0, main run (m = 24 h); only the final N_DAYS_CMP model days are kept
word_fulltime_df = pd.read_pickle(S + f"model0/results/{RUN0}.pkl").iloc[:, -N_DAYS_CMP:]
bt      = np.log10(word_fulltime_df.shift(-1, axis=1) / word_fulltime_df)
real_bt = np.log10(hashtag_df.shift(-1, axis=1) / hashtag_df)

In [ ]:
plot_realdata_usage(hashtag_df, newfigure=True, c = 'C0', errorbar=True, raw = False, marker = 'o')
plot_simulated_usage(word_fulltime_df, c = 'C1', newfigure=False, 
                        day_interval = 1, errorbar=True, raw = False, marker = 's')
plt.gca().xaxis.set_minor_locator(LogLocator(subs='all', numticks=100))
plt.grid(False)
save('model0_usage.pdf')

In [ ]:
plt.figure(figsize=(8, 6))
logbt_error  = compare_log_growth_pdf(bt, real_bt, real_label = 'real', 
                                      simu_label = 'simulation', simu_line=True)
plt.tight_layout()
save('model0_growth_rate.pdf')

In [ ]:
# Fig. 4(c): Model 0. The empirical series is shown without a fit; the simulation is fitted.
plt.figure(figsize=(8, 6))
plot_adopter_scaling(*adopter_pairs(hashtag_df, emp_new), label='real', c='C0', marker='o', fit=False, **FIT)
plot_adopter_scaling(*sim_pair('model0', RUN0), label='simulation', c='C1', marker='s',
                     fit=True, alpha_in_label=False, **FIT)
save('model0_trend_follow_v2.pdf')

In [ ]:
plt.figure(figsize=(8, 6))
error_real = next_day_analysis_new(hashtag_df, logy = True, minvalue=0, label = 'real', c = 'C0', return_=True, 
                                   min_sample_size=1, error_line=False)
error_simu = next_day_analysis_new(word_fulltime_df, logy = True, minvalue=0, label = 'simulation', 
                                   c = 'C1', return_=True, fit_line = False, min_sample_size=1, marker = 's')
save('model0_deactivation.pdf')

In [ ]:
# Fig. 4(e): growth-rate distribution of Model 0 on randomised networks (m = 24 h)
plt.figure(figsize=(8, 6))
net_ls  = ["real_to_random", "real_keep_out_degree", "real_keep_in_degree"]
name_ls = ["random", "keep out", "keep in"]
markers = ['s', '^', 'v']
for cnt, net in enumerate(net_ls):
    df_net = pd.read_pickle(S + f"model0/results/N397369_T30_HOUR93626_pn0.0218_constdk_c1_{net}_net_m24_steps_SEED1.pkl").iloc[:, -N_DAYS_CMP:]
    bt_net = np.log10(df_net.shift(-1, axis=1) / df_net)
    compare_log_growth_pdf(bt_net, real_bt, bins=31, real_bar=(cnt == 0),
                           simu_label=name_ls[cnt], simu_c=f'C{cnt+1}', simu_line=True, simu_marker=markers[cnt],
                           real_label='real' if cnt == 0 else None)
plt.legend(fontsize=20, ncol=1, loc=3, frameon=False, bbox_to_anchor=(0.625, 0.60))
plt.tight_layout()
save('model0_growth_rate_net_comparison.pdf')

In [ ]:
# Fig. 4(f): growth-rate distribution of Model 0 for different memory windows m (hours)
plt.figure(figsize=(8, 6))
markers = ['s', '^', 'v']
for cnt, m in enumerate([24, 72, 120]):
    df_m = pd.read_pickle(S + f"model0/results/N397369_T30_HOUR93626_pn0.0218_constdk_c1_real_net_m{m}_steps_SEED1.pkl").iloc[:, -N_DAYS_CMP:]
    bt_m = np.log10(df_m.shift(-1, axis=1) / df_m)
    compare_log_growth_pdf(bt_m, real_bt, bins=31, real_bar=True,
                           simu_label=rf'$m={m}$', simu_c=f'C{cnt+1}', simu_line=True, simu_marker=markers[cnt],
                           real_label='real' if cnt == 0 else None)
plt.legend(fontsize=20, ncol=1, loc=3, frameon=False, bbox_to_anchor=(0.635, 0.60))
plt.tight_layout()
save('model0_growth_rate_m_comparison.pdf')

## Fig. 5 — effective contagiousness

Needs the 10-minute LCC count tables and `model0_expected_counts_10min.pkl`, the expected counts recomputed with the past-only m = 24 exposure window (`analysis/recompute_expected_counts.py`).

In [ ]:
hashtag_df_hour_lcc = pd.read_pickle(LCC10MIN).fillna(0)        # observed counts x_k(r), 10-minute windows, LCC users

tag_likely_df_hour = pd.read_pickle(R + 'model0_expected_counts_10min.pkl')
tag_likely_df_hour = tag_likely_df_hour.reindex(columns=hashtag_df_hour_lcc.columns)  # align columns; windows not computed become NaN and are removed by the >= 30 filter below
print(len(tag_likely_df_hour))
min_length = 2
min_model_value = 30
length = (tag_likely_df_hour >= min_model_value).sum(axis = 1)
sampled_tags = length[(length>=min_length)]
tag_likely_df_hour = tag_likely_df_hour.loc[sampled_tags.index, :] # keep hashtags with enough reliable windows

# Optional de-noising (posts of non-followed users treated as noise). The paper uses the
# raw counts, so the noise table is only read when DENOISE is switched on.
DENOISE = False
if DENOISE:
    hashtag_df_noise_lcc = pd.read_pickle(NOISE10MIN).fillna(0)
    hashtag_df_noise_lcc = hashtag_df_noise_lcc[hashtag_df_noise_lcc.index.isin(tag_likely_df_hour.index)]
    hashtag_df_hour_lcc_denoise = hashtag_df_hour_lcc - hashtag_df_noise_lcc
    diffu_df = hashtag_df_hour_lcc_denoise.loc[tag_likely_df_hour.index, :] / tag_likely_df_hour
else:
    diffu_df = hashtag_df_hour_lcc.loc[tag_likely_df_hour.index, :] / tag_likely_df_hour
diffu_df = diffu_df.loc[sampled_tags.index, :]
diffu_df = diffu_df.replace([np.inf, -np.inf], np.nan)
diffu_df[tag_likely_df_hour < min_model_value] = np.nan # unreliable windows -> NaN
print(diffu_df.shape)

In [ ]:
name = 'save_ibaraki'
visulize_diffusion_series_paper(hashtag_df_hour_lcc, tag_likely_df_hour, diffu_df, name=name)
save(f'real_contagiousness_series_{name}.pdf')

In [ ]:
min_sample = 3
nbins = 50
name = 'save_ibaraki'
result_bined, result, slope, intercept, x_intersect = potention_analysis_paper(diffu_df.iloc[:, 24*6:],
                tag_name=name, onesample=True,
                nbins=nbins, min_sample=min_sample,
                plot_scatter=True, regression=True, show_mu = False,
                return_=True
                )
plt.xlim(-0.5, 6)
plt.ylim(-1.0, 1.0)
save(f'real_contagiousness_ou_estimation_{name}.pdf')

In [ ]:
# Fig. 5(c): pooled drift of d_k(r) over all reliable hashtags
min_sample = 10
nbins = 50
result_bined, result, slope, intercept, x_intersect = potention_analysis_paper(
                diffu_df.iloc[:, 24*6:],
                top_k=diffu_df.shape[0], nbins=nbins, min_sample=min_sample,
                onesample=False, tag_name='',
                plot_scatter=False, 
                regression=True,
                show_mu=False, 
                return_=True)
plt.legend(fontsize=20, loc = 'lower left', framealpha=0)
save(f'real_contagiousness_ou_estimation_all.pdf')

In [ ]:
# Fig. 5(d): potential U(d) integrated from the pooled drift
plot_potential(result, result_bined)
save(f'real_contagiousness_potential_all.pdf')

## Fig. 6 — Model 1

All six panels come from the single Model 1 run. Run the loader cell first.

In [ ]:
# Model 1, main run (m = 84 h, theta = 0.007, d0 = 13); only the final N_DAYS_CMP model days are kept
word_fulltime_df = pd.read_pickle(S + f"model1/results/{RUN1}.pkl").iloc[:, -N_DAYS_CMP:]
bt      = np.log10(word_fulltime_df.shift(-1, axis=1) / word_fulltime_df)
real_bt = np.log10(hashtag_df.shift(-1, axis=1) / hashtag_df)

In [ ]:
plot_realdata_usage(hashtag_df, newfigure=True, c = 'C0', errorbar=True, raw = False, marker = 'o')
plot_simulated_usage(word_fulltime_df, c = 'C1', newfigure=False, 
                        day_interval = 1, errorbar=True, raw = False, marker = 's')
plt.gca().xaxis.set_minor_locator(LogLocator(subs='all', numticks=100))
plt.grid(False)
save('model1_usage.pdf')

In [ ]:
plt.figure(figsize=(8, 6))
logbt_error  = compare_log_growth_pdf(bt, real_bt, real_label = 'real', 
                                      simu_label = 'simulation', simu_line=True)
plt.tight_layout()
save('model1_growth_rate.pdf')

In [ ]:
paras = growthRateDistribution(word_fulltime_df.fillna(0).T,
                               stationaryPoint = 0, base = 10, step = 0.5, 
                               minmal_sample = 80, 
                               cumulated_norm = False,
                               plot_jump_sample = 1, 
                               show_detail = False)
plt.yticks([10**(-i) for i in range(6)])
save('model1_growth_rate_conditional_cdf.pdf', pad=0.1)

In [ ]:
plt.figure(figsize=(8, 6))
std_ge_result_real, std_ne_result_real = growth_rate_scaling_errorbar_prepare_data(hashtag_df.fillna(0).T, 
                                        stationaryPoint = 0, base = 20, step = 0.5, minmal_sample=30, show_detail = False)
growth_rate_scaling_errorbar_plot(std_ge_result_real, std_ne_result_real, label = ', real', c = ['C0', 'C0'])
std_ge_result_sim, std_ne_result_sim = growth_rate_scaling_errorbar_prepare_data(word_fulltime_df.fillna(0).T, 
                                        stationaryPoint = 0, base = 20, step = 0.5, minmal_sample=30, show_detail = False)
growth_rate_scaling_errorbar_plot(std_ge_result_sim, std_ne_result_sim, label = ', simulation', c = ['C1', 'C1'])
plt.legend(fontsize=19, frameon = False, loc = 3, bbox_to_anchor=(-0.055, 0.565))
plt.tight_layout()
save('model1_growth_rate_scaling_errorbar.pdf')

In [ ]:
# Fig. 6(e): Model 1. The empirical series is shown without a fit; the simulation is fitted.
plt.figure(figsize=(8, 6))
plot_adopter_scaling(*adopter_pairs(hashtag_df, emp_new), label='real', c='C0', marker='o', fit=False, **FIT)
plot_adopter_scaling(*sim_pair('model1', RUN1), label='simulation', c='C1', marker='s',
                     fit=True, alpha_in_label=False, **FIT)
save('model1_trend_follow_v2.pdf')

In [ ]:
plt.figure(figsize=(8, 6))
error_real = next_day_analysis_new(hashtag_df, logy = True, minvalue=0, label = 'real', c = 'C0', return_=True, 
                                   min_sample_size=1, error_line=False)
error_simu = next_day_analysis_new(word_fulltime_df, logy = True, minvalue=0, label = 'simulation', 
                                   c = 'C1', return_=True, fit_line = False, min_sample_size=1, 
                                   error_line=False, marker = 's')
save('model1_deactivation.pdf')

## Fig. 7 — random multiplicative process

Driven by the growth-rate pool of the Model 1 run, so run the Fig. 6 loader cell first.

In [ ]:
# RMP: growth rates drawn from the Model 1 pool per size bin; new hashtags injected as lognormal(mu, std)
base, step, max_exponent = 10, 0.5, 4.5
mu, std, del0 = -0.9, 1.8, False
plt.figure(figsize=(8, 6))
result, growth_rate_ls = random_simulation(word_fulltime_df, days = 500, base = base, step=step, max_exponent=max_exponent, 
                                           lambda_=1, del0 = del0, noise = 'lognormal', 
                                           mu = mu, std = std)
plot_realdata_usage(hashtag_df.iloc[:, -7:], newfigure=True, c = 'k', errorbar=True, raw = False, marker = 'o')
plot_simulated_usage(pd.DataFrame(result.T).iloc[:, -7:], newfigure=False, 
                        day_interval = 1, errorbar=True, raw = False, label = 'RMP simulation', marker='s')
plt.gca().xaxis.set_minor_locator(LogLocator(subs='all', numticks=100))
plt.tight_layout()
plt.grid(False)
save('model1_rmp_simulation.pdf')

In [ ]:
min_length = 3
max_lag = 5
# real bt acf
acf_real_df = batch_autocorr((hashtag_df.shift(-1, axis=1)/hashtag_df).iloc[:, :-1].dropna(how = 'all', axis=0), min_length = min_length, max_lags = max_lag)
acf_real_df.dropna(how = 'all', inplace=True, axis=0)
acf_real_df['lag_0'] = 1
acf_real_df = acf_real_df[[f"lag_{i}" for i in range(6)]]

# abm simulated bt acf

acf_abm_df = batch_autocorr((word_fulltime_df.shift(-1, axis=1)/word_fulltime_df).dropna(how = 'all').iloc[:, :-1], min_length = min_length, max_lags = max_lag)
acf_abm_df.dropna(how = 'all', inplace=True, axis=0)
acf_abm_df['lag_0'] = 1
acf_abm_df = acf_abm_df[[f"lag_{i}" for i in range(6)]]

# rmp simulated bt acf with abm simulated time series
hashtag_df_rmp = pd.DataFrame(result[-7:, :]).T
bt_rmp = (hashtag_df_rmp.shift(-1, axis=1)/hashtag_df_rmp)
acf_rmp_df = batch_autocorr(bt_rmp.iloc[:, :-1].dropna(how = 'all', axis=0), min_length = min_length, max_lags = max_lag)
acf_rmp_df.dropna(how = 'all', inplace=True, axis=0)
acf_rmp_df['lag_0'] = 1
acf_rmp_df = acf_rmp_df[[f"lag_{i}" for i in range(6)]]

In [ ]:
markers = ['o', 's', '^']
labels = ['real', 'ABM simulation', 'RMP simulation']
plt.figure(figsize=(8, 6))
for i, df in enumerate([acf_real_df, acf_abm_df, acf_rmp_df]):
    q1 = df.quantile(0.25, axis=0)
    q2 = df.quantile(0.5, axis=0)
    q3 = df.quantile(0.75, axis=0)
    plt.errorbar(range(6), q2, yerr=[q2 - q1, q3 - q2], 
                capsize=8, fmt = markers[i], markersize = 12, ecolor=f"C{i}", 
                markeredgecolor = f"C{i}", color = 'w', capthick=2.5, linewidth=2.5,
                label= f'[Q1, Q3] ({labels[i]})')
plt.xticks(range(6), range(6))
plt.xlabel('Lag')
plt.ylabel('ACF');
plt.legend(fontsize = 20, frameon=False)
save(f'model1_rmp_acf.pdf')

## Fig. 9a — sampled trajectories of the effective contagiousness

Reads `diffu_value/` of the Model 1 run. The simulation records hashtags with `tag % 10000 == 0`, giving about 150 trajectories.

In [ ]:
# one line per sampled hashtag: tag, d(0), d(1), ...  (one value per 10-minute update)
with open(S + f"model1/diffu_value/sample{RUN1}.txt") as f:
    lines = f.readlines()
tag_name = []
processed_data = []
for i, line in enumerate(lines):
    dks = np.array(list(map(float, line.strip().split(','))))
    processed_data.append(dks[1:])
    tag_name.append(int(dks[0]))
test = pd.DataFrame(processed_data, index=tag_name)

plt.figure(figsize=(8, 6))
for i in range(50):
    tmp = test.iloc[i, :]
    plt.plot(tmp[tmp>0], label = f'Tag {i}')
    plt.plot(tmp[tmp<0], c = 'gray', alpha = 0.3)
plt.axhline(y=0, color='red', linestyle='--', linewidth = 3, alpha=0.8)
maxday = 7
for i in range(0, 24 * 6 * maxday, 6 * 24): plt.axvline(x=i, color='gray', linestyle='--', alpha=0.5)
plt.xlim([-10, 24 * 6 * maxday + 10])
plt.xlabel(r'$r$')
plt.ylabel(r'$d_k(r)$');
plt.tight_layout();
save('model1_diffusion_series_example.pdf')